# Logistic Regression
**Jennifer Eigo - University of Connecticut - Dept. of Operations and Information Management**

-------------------------------------
Regression models are commonly used models in data science. They are very explainable so you not only build a model, but you also learn more about relationships in the data. Logistic regressions predict a categorical target variable.


# Environment Setup

In [ ]:
# import modules

import pandas as pd # for data viz and wrangling
import numpy as np # for 'numeric python'
import matplotlib.pyplot as plt # for data viz (more complex than pylab)
import seaborn as sns
from pylab import * # for data viz (import * means 'import all of the functions')
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn import metrics
from scipy import stats
import statsmodels.api as sm


In [ ]:
# mount your google drive
from google.colab import drive
drive.mount('/content/drive')

#Load the Data

To practice logistic regression, we need a dataset that has a categorical target.  The Universal Bank dataset can be used to predict if a customer will accept an offer of a personal loan.  Each row represents a bank customer and the target variable is called Personal Loan.

In [ ]:
# read data
# on the lefthand side, navigate to your data and copy the path
df = pd.read_csv('/content/drive/MyDrive/OPIM 5604 Python/Module 6/UniversalBank.csv')

In [ ]:
# shape
# shows how many rows and columns
# this sample has 5000 rows and 14 columns
df.shape

In [ ]:
# Preview
print(df.head())

In [ ]:
# list the columns with the data types
print(df.info())

Looks like a mix of categorical and continuous predictors.  We should  change our target variable to 1s and 0s.  And let's make dummy variables for Education.  We should drop Zip Code because it has too many categories.  

In [ ]:
# Show a histogram of 'Personal Loan' counts
plt.figure(figsize=(8, 6))
sns.histplot(data=df, x='Personal Loan', bins=2) # bins=2 for two categories (0 and 1)
plt.title('Distribution of Personal Loan')
plt.xlabel('Personal Loan')
plt.ylabel('Count')
plt.xticks([0, 1]) # Set x-axis ticks at 0 and 1
plt.show()

# Display counts
personal_loan_counts = df['Personal Loan'].value_counts()
print("\nCounts of Personal Loan:")
print(personal_loan_counts)

In [ ]:
# Convert 'Personal Loan' to numerical (Yes=1, No=0) using map
df['Personal Loan'] = df['Personal Loan'].map({'Yes': 1, 'No': 0})

# Display the first few rows and unique values to see the changes in the target variable
print(df.head())
print("\nUnique values after conversion:")
print(df['Personal Loan'].unique())

In [ ]:
# Look at counts of 0 and 1 values for Personal Loan to check our work
personal_loan_counts = df['Personal Loan'].value_counts()
print("Counts of Personal Loan (0s and 1s):")
print(personal_loan_counts)

In [ ]:
# Drop the 'ZIP Code' column
df = df.drop('ZIP Code', axis=1)

In [ ]:
# Show a histogram/bar plot of 'Education' with counts
plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='Education')
plt.title('Distribution of Education Levels')
plt.xlabel('Education Level')
plt.ylabel('Count')
plt.show()

# Display counts and percentages
education_counts = df['Education'].value_counts()
education_percentages = df['Education'].value_counts(normalize=True) * 100

print("\nCounts of Education Levels:")
print(education_counts)

print("\nPercentages of Education Levels:")
print(education_percentages)

In [ ]:
# Create dummy variables for 'Education'
df = pd.get_dummies(df, columns=['Education'], drop_first=True)

# Display the first few rows to see the changes
print(df.head())

Now let's partition the data to get it ready for modeling. We define Personal Loan as the target variable and the rest of the columns as our predictor variables.  We will do a 50/30/20 split.  In the code we first separate out the 20% for test, then we split the remaining portion into training and validation.

In [ ]:
from sklearn.model_selection import train_test_split

# Define the target variable
target = 'Personal Loan'
# Define features by dropping the target variable and the 'ID' column
features = df.drop([target, 'ID'], axis=1)
target_variable = df[target]

# Split the data into 80% training and 20% test
X_train, X_test, y_train, y_test = train_test_split(features, target_variable, test_size=0.2, random_state=42)

# Split the 80% training data into 50% training and 30% validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.375, random_state=42) # 0.375 * 0.8 = 0.3

# Print the shapes of the resulting datasets
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Combine the features and target for plotting
train_data = X_train.copy()
train_data['Personal Loan'] = y_train

val_data = X_val.copy()
val_data['Personal Loan'] = y_val

test_data = X_test.copy()
test_data['Personal Loan'] = y_test

# Plot the distribution of 'Personal Loan' in each partition
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

sns.countplot(data=train_data, x='Personal Loan', ax=axs[0])
axs[0].set_title('Training Set - Personal Loan Distribution')
axs[0].set_xlabel('Personal Loan')
axs[0].set_ylabel('Count')
axs[0].set_xticks([0, 1])
axs[0].set_xticklabels(['No', 'Yes'])

sns.countplot(data=val_data, x='Personal Loan', ax=axs[1])
axs[1].set_title('Validation Set - Personal Loan Distribution')
axs[1].set_xlabel('Personal Loan')
axs[1].set_ylabel('Count')
axs[1].set_xticks([0, 1])
axs[1].set_xticklabels(['No', 'Yes'])


sns.countplot(data=test_data, x='Personal Loan', ax=axs[2])
axs[2].set_title('Test Set - Personal Loan Distribution')
axs[2].set_xlabel('Personal Loan')
axs[2].set_ylabel('Count')
axs[2].set_xticks([0, 1])
axs[2].set_xticklabels(['No', 'Yes'])

plt.tight_layout()
plt.show()

# Print counts and percentages for each partition
print("Training set Personal Loan counts and percentages:")
train_counts = train_data['Personal Loan'].value_counts()
train_percentages = train_data['Personal Loan'].value_counts(normalize=True) * 100
print(train_counts)
print(train_percentages.round(2)) # Round percentages to 2 decimal places


print("\nValidation set Personal Loan counts and percentages:")
val_counts = val_data['Personal Loan'].value_counts()
val_percentages = val_data['Personal Loan'].value_counts(normalize=True) * 100
print(val_counts)
print(val_percentages.round(2)) # Round percentages to 2 decimal places


print("\nTest set Personal Loan counts and percentages:")
test_counts = test_data['Personal Loan'].value_counts()
test_percentages = test_data['Personal Loan'].value_counts(normalize=True) * 100
print(test_counts)
print(test_percentages.round(2)) # Round percentages to 2 decimal places

# Logistic Regression

Since we have a categorical target we need to build a logistic regression model.

##Train the Model

First we fit the model.

In [ ]:
# Initialize and train the Logistic Regression model
logistic_model = LogisticRegression(random_state=42)
logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Then we make predictions using the formula that was fitted on the training data.

In [ ]:
# Make predictions (classes) on the training, validation, and test sets
y_train_pred = logistic_model.predict(X_train)
y_val_pred = logistic_model.predict(X_val)
y_test_pred = logistic_model.predict(X_test)

# Get predicted probabilities for the training, validation, and test sets
y_train_proba = logistic_model.predict_proba(X_train)[:, 1] # Probability of the positive class (1)
y_val_proba = logistic_model.predict_proba(X_val)[:, 1]
y_test_proba = logistic_model.predict_proba(X_test)[:, 1]

print("Predictions (classes and probabilities) made successfully on training, validation, and test sets.")

Let's take a look at the predictions.  Notice how the model predicts probabilities which are then converted to classes based off a default cutoff of 0.5.

In [ ]:
# Show actuals, probabilities, and predictions for the first 10 rows of the test set in a DataFrame
import pandas as pd

results_df = pd.DataFrame({
    'Actual': y_test.head(10).tolist(),
    'Predicted Probability (Personal Loan=1)': y_test_proba[:10].tolist(),
    'Predicted Class': y_test_pred[:10].tolist()
})

print("Actuals, Probabilities, and Predicted Personal Loan (First 10 rows of Test Set):")
display(results_df)

Since none of the rows above have a probability near 0.5, let's check again and be more specific.

In [ ]:
# Filter the test set to show rows where predicted probability is between 0.45 and 0.55
uncertain_predictions_indices = (y_test_proba >= 0.45) & (y_test_proba <= 0.55)

# Get the corresponding actual values, predicted probabilities, and predicted classes
uncertain_actual = y_test[uncertain_predictions_indices]
uncertain_proba = y_test_proba[uncertain_predictions_indices]
uncertain_pred = y_test_pred[uncertain_predictions_indices]

# Create a DataFrame to display these results
uncertain_results_df = pd.DataFrame({
    'Actual': uncertain_actual,
    'Predicted Probability (Personal Loan=1)': uncertain_proba,
    'Predicted Class': uncertain_pred
})

# Display the first 10 rows from this filtered DataFrame
print("Test Set Rows with Predicted Probability between 0.45 and 0.55 (First 10):")
display(uncertain_results_df.head(10))

# Also show the number of such cases
print(f"\nNumber of cases with predicted probability between 0.45 and 0.55 in the test set: {len(uncertain_results_df)}")

Now we can see the 0.5 cutoff in action.  

Let's also take a look at the coefficients.  Just like before we can learn about our predictors by assessing the sign of the coefficients.  Positive coefficients make people more likely to accept the loan.  Negative coefficients make people less likely to accept the loan.  

In [ ]:
# Show the coefficients of the logistic regression model, including the intercept
feature_names = X_train.columns
coefficients = logistic_model.coef_[0]
intercept = logistic_model.intercept_[0]

# Create a DataFrame for coefficients and feature names
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})

# Add the intercept to the DataFrame
intercept_df = pd.DataFrame({'Feature': ['Intercept'], 'Coefficient': [intercept]})
coef_df = pd.concat([intercept_df, coef_df], ignore_index=True)

print("\nCoefficients (including Intercept) with Feature Names:")
print(coef_df)

In [ ]:
# Show the regression formula
# The formula for logistic regression is: log(p / (1 - p)) = b0 + b1*x1 + b2*x2 + ... + bn*xn
# Rearranging for p: p = 1 / (1 + exp(-(b0 + b1*x1 + b2*x2 + ... + bn*xn)))

intercept = logistic_model.intercept_[0]

linear_combination = f"{intercept:.4f}"
for i, feature in enumerate(feature_names):
    linear_combination += f" + {coefficients[i]:.4f}*{feature}"

formula = f"p = 1 / (1 + exp(-({linear_combination})))"

print("Logistic Regression Formula (with probability on the left):")
print(formula)

##Model Performance

It's always important to look at the confusion matrixes so let's start there.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate confusion matrix for the training set
cm_train = confusion_matrix(y_train, y_train_pred)
print("Confusion Matrix - Training Set:")
print(cm_train)

# Calculate confusion matrix for the validation set
cm_val = confusion_matrix(y_val, y_val_pred)
print("\nConfusion Matrix - Validation Set:")
print(cm_val)

# Calculate confusion matrix for the test set
cm_test = confusion_matrix(y_test, y_test_pred)
print("\nConfusion Matrix - Test Set:")
print(cm_test)

# Optional: Visualize Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=axes[0], annot_kws={"size": 12})
axes[0].set_title('Confusion Matrix - Training Set')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', ax=axes[1], annot_kws={"size": 12})
axes[1].set_title('Confusion Matrix - Validation Set')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', ax=axes[2], annot_kws={"size": 12})
axes[2].set_title('Confusion Matrix - Test Set')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.show()

Now let's look at the classification report which shows the important model performance metrics.

In [ ]:
from sklearn.metrics import classification_report

# Show classification report for the training set
print("Classification Report - Training Set:")
print(classification_report(y_train, y_train_pred))

# Show classification report for the validation set
print("\nClassification Report - Validation Set:")
print(classification_report(y_val, y_val_pred))

# Show classification report for the test set
print("\nClassification Report - Test Set:")
print(classification_report(y_test, y_test_pred))

The lift cureve can be helpful if we are looking for a specific improvement over our current response rate.

In [ ]:
def plot_lift_curve(y_true, y_proba, title):
    """Plots the lift curve for a binary classification model."""
    # Ensure y_true and y_proba are numpy arrays
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    # Sort the data by predicted probability in descending order
    sorted_indices = np.argsort(y_proba)[::-1]
    y_true_sorted = y_true[sorted_indices]

    # Calculate the cumulative number of positive instances
    cumulative_positives = np.cumsum(y_true_sorted)

    # Calculate the cumulative number of instances
    cumulative_instances = np.arange(1, len(y_true) + 1)

    # Calculate the lift
    # Avoid division by zero by checking if cumulative_instances is 0
    lift = np.where(cumulative_instances > 0, cumulative_positives / cumulative_instances / (np.sum(y_true) / len(y_true)), 0)


    plt.plot(cumulative_instances, lift, marker='.', linestyle='none', markersize=4, label='Model Lift')
    plt.plot([0, len(y_true)], [1, 1], linestyle='--', color='red', label='Random Lift') # Random lift is always 1

    plt.xlabel("Number of Instances (Sorted by Predicted Probability)")
    plt.ylabel("Lift")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.ylim([0, max(lift) * 1.1 if len(lift) > 0 else 2]) # Adjust y-axis limit
    plt.xlim([0, len(y_true)]) # Adjust x-axis limit


# Plot lift curve for each partition
plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plot_lift_curve(y_train, y_train_proba, 'Lift Curve - Training Set')

plt.subplot(1, 3, 2)
plot_lift_curve(y_val, y_val_proba, 'Lift Curve - Validation Set')

plt.subplot(1, 3, 3)
plot_lift_curve(y_test, y_test_proba, 'Lift Curve - Test Set')

plt.tight_layout()
plt.show()

Lastly, let's look at the AUC for each partition.

In [ ]:
# Calculate ROC curve and AUC for the training set
fpr_train, tpr_train, thresholds_train = roc_curve(y_train, y_train_proba)
roc_auc_train = auc(fpr_train, tpr_train)

# Calculate ROC curve and AUC for the validation set
fpr_val, tpr_val, thresholds_val = roc_curve(y_val, y_val_proba)
roc_auc_val = auc(fpr_val, tpr_val)

# Calculate ROC curve and AUC for the test set
fpr_test, tpr_test, thresholds_test = roc_curve(y_test, y_test_proba)
roc_auc_test = auc(fpr_test, tpr_test)

# Plot ROC curves
plt.figure(figsize=(10, 8))
plt.plot(fpr_train, tpr_train, color='darkorange', lw=2, label='Training ROC curve (AUC = %0.2f)' % roc_auc_train)
plt.plot(fpr_val, tpr_val, color='green', lw=2, label='Validation ROC curve (AUC = %0.2f)' % roc_auc_val)
plt.plot(fpr_test, tpr_test, color='blue', lw=2, label='Test ROC curve (AUC = %0.2f)' % roc_auc_test)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guessing (AUC = 0.50)') # Diagonal line for random guessing
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

print(f"Training AUC: {roc_auc_train:.4f}")
print(f"Validation AUC: {roc_auc_val:.4f}")
print(f"Test AUC: {roc_auc_test:.4f}")

## Variable Selection

We should always check the statistical significance of our predictors with our P values.

In [ ]:
# Convert boolean columns to integers (0s and 1s) for statsmodels
X_train_sm = X_train.astype(int)

# Add a constant (intercept) to the features for statsmodels
X_train_sm = sm.add_constant(X_train_sm)

# Fit the logistic regression model using statsmodels
logit_model = sm.Logit(y_train, X_train_sm)
result = logit_model.fit()

# Show the model summary, which includes p-values
print(result.summary())

Looks like we could improve the model by removing Age, Income, and Securities Account to start.  Even though Education_Professional has a high P value, Education_Undergrad does not so we will keep that in.  Those numbers are telling us that Professional is not that different from Graduate (the refernce category that was dropped when the dummy variables were formed), while Undergrad is very different.  To address this we could do some additoinal feature engineering and change the education column into two categories - one for Undergrad and one for Graduate/Profesional combined.  Give it a try on your own!

##Interaction Terms

Sometimes variables have a combined effect on a prediction in addition to the separate effect of each.  In this data, think about the effect of income and family size.  We already saw that both of the variables have a positive impact of people accepting the personal loan.  But when considered together, these numbers can have different meaning.  For instance, a single person with an income of \$100,000 probably has less need for the loan than a family of 4 living on a household income of \$100,000. Interaction terms can be added to our model to help capture these patterns and improve performance.

In [ ]:
# Create the interaction term between Family and Income
X_train_interact = X_train.copy()
X_val_interact = X_val.copy()
X_test_interact = X_test.copy()

X_train_interact['Family_Income_Interaction'] = X_train_interact['Family'] * X_train_interact['Income']
X_val_interact['Family_Income_Interaction'] = X_val_interact['Family'] * X_val_interact['Income']
X_test_interact['Family_Income_Interaction'] = X_test_interact['Family'] * X_test_interact['Income']

# Initialize and train the Logistic Regression model with the interaction term
logistic_model_interact = LogisticRegression(random_state=42, max_iter=1000) # Increased max_iter to address potential convergence issues
logistic_model_interact.fit(X_train_interact, y_train)

print("Logistic Regression model with Family and Income interaction term trained successfully.")

# Make predictions with the new model
y_train_pred_interact = logistic_model_interact.predict(X_train_interact)
y_val_pred_interact = logistic_model_interact.predict(X_val_interact)
y_test_pred_interact = logistic_model_interact.predict(X_test_interact)

# Get predicted probabilities with the new model
y_train_proba_interact = logistic_model_interact.predict_proba(X_train_interact)[:, 1]
y_val_proba_interact = logistic_model_interact.predict_proba(X_val_interact)[:, 1]
y_test_proba_interact = logistic_model_interact.predict_proba(X_test_interact)[:, 1]

print("Predictions with interaction term made successfully.")

Let's check to see if the new interaction term is statistically significant.

In [ ]:
# Convert boolean columns to integers (0s and 1s) for statsmodels, if any
X_train_interact_sm = X_train_interact.astype(int)

# Add a constant (intercept) to the features for statsmodels
X_train_interact_sm = sm.add_constant(X_train_interact_sm)

# Fit the logistic regression model using statsmodels
logit_model_interact = sm.Logit(y_train, X_train_interact_sm)
result_interact = logit_model_interact.fit()

# Show the model summary, which includes p-values
print(result_interact.summary())

Yep, it is!  Look at that very low P value of 0.000

How does performance change with the inclusion of the interaction term?

In [ ]:
# Calculate confusion matrix for the training set with interaction term
cm_train_interact = confusion_matrix(y_train, y_train_pred_interact)
print("Confusion Matrix - Training Set (with Interaction Term):")
print(cm_train_interact)

# Calculate confusion matrix for the validation set with interaction term
cm_val_interact = confusion_matrix(y_val, y_val_pred_interact)
print("\nConfusion Matrix - Validation Set (with Interaction Term):")
print(cm_val_interact)

# Calculate confusion matrix for the test set with interaction term
cm_test_interact = confusion_matrix(y_test, y_test_pred_interact)
print("\nConfusion Matrix - Test Set (with Interaction Term):")
print(cm_test_interact)

# Optional: Visualize Confusion Matrices with Interaction Term
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm_train_interact, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix - Training Set (with Interaction Term)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_val_interact, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Confusion Matrix - Validation Set (with Interaction Term)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

sns.heatmap(cm_test_interact, annot=True, fmt='d', cmap='Blues', ax=axes[2])
axes[2].set_title('Confusion Matrix - Test Set (with Interaction Term)')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.show()

Looks better!  Our FP and FN have reduced in all three partitions.  This should improve all of our performance metrics.

In [ ]:
# Show classification report for the training set with interaction term
print("Classification Report - Training Set (with Interaction Term):")
print(classification_report(y_train, y_train_pred_interact))

# Show classification report for the validation set with interaction term
print("\nClassification Report - Validation Set (with Interaction Term):")
print(classification_report(y_val, y_val_pred_interact))

# Show classification report for the test set with interaction term
print("\nClassification Report - Test Set (with Interaction Term):")
print(classification_report(y_test, y_test_pred_interact))

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Calculate ROC curve and AUC for the training set with interaction term
fpr_train_interact, tpr_train_interact, thresholds_train_interact = roc_curve(y_train, y_train_proba_interact)
roc_auc_train_interact = auc(fpr_train_interact, tpr_train_interact)

# Calculate ROC curve and AUC for the validation set with interaction term
fpr_val_interact, tpr_val_interact, thresholds_val_interact = roc_curve(y_val, y_val_proba_interact)
roc_auc_val_interact = auc(fpr_val_interact, tpr_val_interact)

# Calculate ROC curve and AUC for the test set with interaction term
fpr_test_interact, tpr_test_interact, thresholds_test_interact = roc_curve(y_test, y_test_proba_interact)
roc_auc_test_interact = auc(fpr_test_interact, tpr_test_interact)

# Plot ROC curves with interaction term
plt.figure(figsize=(10, 8))
plt.plot(fpr_train_interact, tpr_train_interact, color='darkorange', lw=2, label='Training ROC curve (AUC = %0.2f)' % roc_auc_train_interact)
plt.plot(fpr_val_interact, tpr_val_interact, color='green', lw=2, label='Validation ROC curve (AUC = %0.2f)' % roc_auc_val_interact)
plt.plot(fpr_test_interact, tpr_test_interact, color='blue', lw=2, label='Test ROC curve (AUC = %0.2f)' % roc_auc_test_interact)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guessing (AUC = 0.50)') # Diagonal line for random guessing
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve (with Interaction Term)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

print(f"Training AUC (with Interaction Term): {roc_auc_train_interact:.4f}")
print(f"Validation AUC (with Interaction Term): {roc_auc_val_interact:.4f}")
print(f"Test AUC (with Interaction Term): {roc_auc_test_interact:.4f}")

And it has!  Everything has improved a bit with the addition of the interaction term.